# 1. mrp testing - BASE

summary the idea of execution
- 1. Create 4 helper function
    - all_items : find ALL item_needed for MRP logic from BOM table and Order table
    - parents_map: create a single-level BOM for exploding parent net --> parent planned receipt --> parent planned release ---> child gross
    - compute_item levels: na ná multi-level BOM but find the LEVEL for MRP iterative process
    - d
- 2. Logic is
    - 1st: START INITIALIZE the FORMAT
    - 2nd: LOOP: 
        - through BOM level (multi-level) 
            - -> item in BOM level 
                - -> Week-by-week to find gross-to-net-to-PR/PS 
                    - -> item in single-BOM to find child-gross
- 3. FORMAT: a dictionary of Item : Dataframe
    - Dataframe is long format for standard MRP components (gross,Proj On-Hand,net,PR,PL)
    

Không khả thi vì cột projected-on hand mà không biến đổi theo PR/PL thì nếu rounding value quá cao thì ăn cức

vì không xét đến tồn kho giả định (khi hàng của PR/PL về mà không tính vào tồn kho)

In [2]:
# Time-phased MRP example (lot-for-lot), generating planner-style grids
import pandas as pd
from collections import defaultdict, deque

# ----------------------------
# Inputs
# ----------------------------
horizon_weeks = 6  # plan weeks 0..6

# Demand (external)
external_demand = pd.DataFrame([
    {"item": "FP", "week": 1, "qty": 888},
    {"item": "FP", "week": 4, "qty": 100},
    {"item": "FP", "week": 5, "qty": 188},
    {"item": "FP", "week": 6, "qty": 388},
])

# BOM (parent -> component with qty per)
bom = pd.DataFrame([
    {"parent": "FP", "component": "A",     "qty_per": 2},
    {"parent": "FP", "component": "B",     "qty_per": 3},
    {"parent": "A",  "component": "RawX",  "qty_per": 1},
    {"parent": "B",  "component": "RawY",  "qty_per": 2},
])

# Item lead times (weeks)
lead_time = {
    "FP": 1,
    "A": 1,
    "B": 2,
    "RawX": 2,
    "RawY": 1,
}

# On-hand inventory at week 0
on_hand0 = {
    "FP": 0,
    "A": 20,
    "B": 10,
    "RawX": 50,
    "RawY": 0,
}

# Scheduled receipts by item & week (if any open POs/WO exist)
scheduled_receipts = pd.DataFrame([
    {"item": "A", "week": 2, "qty": 199},  # example placeholder; currently none
    {"item": "FP", "week": 1, "qty": 499},
    {"item": "FP", "week": 5, "qty": 299}
])

# Policy: Lot-for-Lot (planned receipt equals net requirement)
lot_sizing = "L4L"




In [16]:
# ----------------------------
# Helpers
# ----------------------------
def all_items(bom: pd.DataFrame, external: pd.DataFrame):
    items = set(external["item"].unique()) 
    items |= set(bom["parent"].unique()) #Union for Sets.
    items |= set(bom["component"].unique())
    return sorted(items)

def parents_map(bom: pd.DataFrame):
    pm = defaultdict(list)
    for _, r in bom.iterrows():
        pm[r["component"]].append((r["parent"], r["qty_per"]))
    return pm

def children_map(bom: pd.DataFrame):
    cm = defaultdict(list)
    for _, r in bom.iterrows():
        cm[r["parent"]].append((r["component"], r["qty_per"]))
    return cm

def compute_item_levels(bom: pd.DataFrame, roots):
    """
    Assign levels for top-down explosion. Root(s) (finished goods) at level 0.
    """
    cm = children_map(bom)
    level = {}
    q = deque()
    for r in roots:
        level[r] = 0
        q.append(r)
    while q:
        p = q.popleft()
        for c, _ in cm.get(p, []):
            if c not in level or level[c] < level[p] + 1:
                level[c] = level[p] + 1
                q.append(c)
    return level

In [17]:
# ----------------------------
# Initialize time-phased structures
# ----------------------------
items = all_items(bom, external_demand)
weeks = list(range(horizon_weeks + 1))

columns = ["Gross Req", "Scheduled Rcpt", "Proj On-Hand", "Net Req", "Planned Rcpt", "Planned Rel"]
tp = {it: pd.DataFrame(0, index=weeks, columns=columns) for it in items}

# seed external gross requirements
for _, r in external_demand.iterrows():
    w = int(r["week"])
    if w in tp[r["item"]].index:
        tp[r["item"]].loc[w, "Gross Req"] += int(r["qty"])

# seed scheduled receipts
for _, r in scheduled_receipts.iterrows():
    w = int(r["week"])
    if w in tp[r["item"]].index:
        tp[r["item"]].loc[w, "Scheduled Rcpt"] += int(r["qty"])

# set initial on-hand at week 0 BEFORE covering anything
for it in items:
    tp[it].loc[0, "Proj On-Hand"] = on_hand0.get(it, 0)

In [18]:
display(tp)

{'A':    Gross Req  Scheduled Rcpt  Proj On-Hand  Net Req  Planned Rcpt  Planned Rel
 0          0               0            20        0             0            0
 1          0               0             0        0             0            0
 2          0             199             0        0             0            0
 3          0               0             0        0             0            0
 4          0               0             0        0             0            0
 5          0               0             0        0             0            0
 6          0               0             0        0             0            0,
 'B':    Gross Req  Scheduled Rcpt  Proj On-Hand  Net Req  Planned Rcpt  Planned Rel
 0          0               0            10        0             0            0
 1          0               0             0        0             0            0
 2          0               0             0        0             0            0
 3          0               0

In [ ]:
"REMEMBER, WHEN COPYING A LIST, if a copied version change, the ORIGIN change"

"""
dua vao concept tp and pd.df
- trong do tp la dict {item : pd.df}

- tp = df_parent ---> net requirement calculation + release/receipt
- tp = df_child ---> release-to-gross explosion + add new task to tp for loop next level

"""

# ----------------------------
# Planning loop by levels, then by time
# ----------------------------
# Determine "roots": items appearing in external demand
roots = sorted(external_demand["item"].unique())
levels = compute_item_levels(bom, roots)

print(roots)
print(levels)

# Items ordered by level (parents before children)
items_ordered = sorted(items, key=lambda x: levels.get(x, 10**6))
print(items_ordered)
cm = children_map(bom)
pm = parents_map(bom)
print(cm)
print(pm)

for it in items_ordered:
    lt = int(lead_time.get(it, 0))
    parent_df = tp[it]

    # 1. roll through weeks to compute netting requirement & planned receipts/releases
    # ---- 1.1 NETTING REQUIREMENT
    for w in weeks:
        gross = parent_df.loc[w, "Gross Req"]
        sched = parent_df.loc[w, "Scheduled Rcpt"]

        if w == 0:
            onhand_prev = parent_df.loc[0, "Proj On-Hand"]
        else:
            onhand_prev = parent_df.loc[w-1, "Proj On-Hand"]

        # Projected on-hand BEFORE planning receipt
        available = onhand_prev + sched

        # If available covers gross, FULL SUPPLY
        if available >= gross:
            net = 0
            planned_rcpt = 0
            proj_onhand = available - gross
        # If available can not cover gross, PARTIAL SUPPLY
        else:
            net = gross - available
            
            # ---- 1.2 CALC receipted + release
            # Lot-for-Lot planned receipt = net
            planned_rcpt = net
            proj_onhand = 0

        parent_df.loc[w, "Net Req"] = net
        parent_df.loc[w, "Planned Rcpt"] = planned_rcpt
        parent_df.loc[w, "Proj On-Hand"] = proj_onhand

        # Time-phase: release = receipt shifted left by lead time
        rel_week = w - lt
        if planned_rcpt > 0 and rel_week >= 0:
            parent_df.loc[rel_week, "Planned Rel"] += planned_rcpt


    # 2. drive children's gross requirements, BOM explosion
    # based on parent's planned releases
    if it in cm:
        for w in weeks:
            parent_rel = parent_df.loc[w, "Planned Rel"]
            if parent_rel > 0:
                
                #update tp (the task) at child ---> update data for upper processing (iterative)
                for child, qty_per in cm[it]:
                    child_df = tp[child]
                    # Child gross requirement occurs at parent's release week
                    need_week = w
                    if need_week in child_df.index:
                        child_df.loc[need_week, "Gross Req"] += parent_rel * qty_per



['FP']
{'FP': 0, 'A': 1, 'B': 1, 'RawX': 2, 'RawY': 2}
['FP', 'A', 'B', 'RawX', 'RawY']
defaultdict(<class 'list'>, {'FP': [('A', 2), ('B', 3)], 'A': [('RawX', 1)], 'B': [('RawY', 2)]})
defaultdict(<class 'list'>, {'A': [('FP', 2)], 'B': [('FP', 3)], 'RawX': [('A', 1)], 'RawY': [('B', 2)]})


## result

In [22]:
display(tp['FP'])

,Gross Req,Scheduled Rcpt,Proj On-Hand,Net Req,Planned Rcpt,Planned Rel
0,0,0,0,0,0,389
1,888,499,0,389,389,0
2,0,0,0,0,0,0
3,0,0,0,0,0,100
4,100,0,0,100,100,0
5,188,299,111,0,0,277
6,388,0,0,277,277,0


In [23]:
# ----------------------------
# Prepare outputs
# ----------------------------
# Append item label as a column and stack all into one DataFrame
stacked = []
for it in items_ordered:
    out = tp[it].copy()
    out.insert(0, "Item", it)
    out.insert(1, "Week", out.index)
    stacked.append(out.reset_index(drop=True))

result_df = pd.concat(stacked, axis=0, ignore_index=True)


# Save to Excel
excel_path = "output_python_example/1st_mrp_time_phased_example.xlsx"
with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    for it in items_ordered:
        temp = result_df[result_df["Item"] == it].drop(columns=["Item"]).set_index("Week")
        temp.to_excel(writer, sheet_name=it)
    # combined
    result_df.to_excel(writer, sheet_name="All_Items", index=False)

excel_path

'output_python_example/1st_mrp_time_phased_example.xlsx'

# 2. (chưa chạm) - Multiple plant

## Configurable input

In [8]:
# Retry execution with same multi-plant / multi-line MRP with capacity check
import pandas as pd
from collections import defaultdict, deque

# ----------------------------
# Configurable Inputs
# ----------------------------
HORIZON_WEEKS = 8  # planning horizon weeks 0..8

# External demand per plant (MPS): can originate per plant
external_demand = pd.DataFrame([
    {"item": "FP", "week": 4, "qty": 120, "plant": "Plant1"},
    {"item": "FP", "week": 5, "qty": 80, "plant": "Plant2"},
])

# BOM (parent -> component with qty per)
bom = pd.DataFrame([
    {"parent": "FP", "component": "A",     "qty_per": 2},
    {"parent": "FP", "component": "B",     "qty_per": 3},
    {"parent": "A",  "component": "RawX",  "qty_per": 1},
    {"parent": "B",  "component": "RawY",  "qty_per": 2},
])

# Lead times (weeks) by item
lead_time = {
    "FP": 1,
    "A": 1,
    "B": 2,
    "RawX": 2,
    "RawY": 1,
}

# Plants and production lines with weekly capacity in hours
plants = {
    "Plant1": {
        "lines": {"LineA1": 40, "LineA2": 30},  # hours per week
    },
    "Plant2": {
        "lines": {"LineB1": 50},  # single line
    }
}

# Production time per unit (hours per unit) by item when produced in a plant
# For purchased items (raw materials) time_per_unit = 0 (no internal capacity)
time_per_unit = {
    "FP": 0.5,   # 0.5 hours to assemble one FP unit
    "A": 0.2,
    "B": 0.3,
    "RawX": 0.0,  # assumed purchased
    "RawY": 0.0,  # assumed purchased
}

# Mapping: where items can be produced (plant-level). If not listed, treated as purchased.
production_sites = {
    "FP": ["Plant1", "Plant2"],
    "A": ["Plant1"],
    "B": ["Plant2"],
    # RawX/RawY purchased -> no production sites
}
# On-hand inventory by (plant, item) at week 0
on_hand0 = {
    ("Plant1", "FP"): 0,
    ("Plant1", "A"): 20,
    ("Plant1", "B"): 0,
    ("Plant1", "RawX"): 50,
    ("Plant1", "RawY"): 0,
    ("Plant2", "FP"): 0,
    ("Plant2", "A"): 0,
    ("Plant2", "B"): 10,
    ("Plant2", "RawX"): 0,
    ("Plant2", "RawY"): 0,
}
# Scheduled receipts (existing POs/WO) by (plant,item,week)
scheduled_receipts = pd.DataFrame([
    # example: {"plant": "Plant1", "item": "A", "week": 2, "qty": 50},
])

# Lot sizing policy: Lot-for-Lot per plant
lot_sizing = "L4L"




## Helpers function

In [9]:
# ----------------------------
# Helpers
# ----------------------------
def all_items(bom: pd.DataFrame, external: pd.DataFrame):
    items = set(external["item"].unique())
    items |= set(bom["parent"].unique())
    items |= set(bom["component"].unique())
    return sorted(items)

def children_map(bom: pd.DataFrame):
    cm = defaultdict(list)
    for _, r in bom.iterrows():
        cm[r["parent"]].append((r["component"], r["qty_per"]))
    return cm

def compute_levels(bom: pd.DataFrame, roots):
    cm = children_map(bom)
    level = {}
    q = deque()
    for r in roots:
        level[r] = 0
        q.append(r)
    while q:
        p = q.popleft()
        for c, _ in cm.get(p, []):
            if c not in level or level[c] < level[p] + 1:
                level[c] = level[p] + 1
                q.append(c)
    return level



## Intialize time-phased structures (dictionary (key1 : key2 : pd.dataframe) )

In [10]:
# ----------------------------
# Initialize time-phased structures
# ----------------------------

# Create planning buckets per plant-item-week
items = all_items(bom, external_demand)
weeks = list(range(HORIZON_WEEKS + 1))

# Initialize data structures: nested dict plant->item->DataFrame
columns = ["Gross Req", "Scheduled Rcpt", "Proj On-Hand", "Net Req", "Planned Rcpt", "Planned Rel"]
tp = {plant: {it: pd.DataFrame(0, index=weeks, columns=columns) for it in items} for plant in plants.keys()}

# Seed external demand per plant into Gross Req
for _, r in external_demand.iterrows():
    plant = r["plant"]
    if plant in tp and r["item"] in tp[plant]:
        w = int(r["week"])
        tp[plant][r["item"]].loc[w, "Gross Req"] += int(r["qty"])

# Seed scheduled receipts
for _, r in scheduled_receipts.iterrows():
    plant = r["plant"]
    w = int(r["week"])
    tp[plant][r["item"]].loc[w, "Scheduled Rcpt"] += int(r["qty"])

# Set initial on-hand per plant-item at week 0
for (plant, item), qty in on_hand0.items():
    if plant in tp and item in tp[plant]:
        tp[plant][item].loc[0, "Proj On-Hand"] = qty



## Start planning loop

In [14]:
# Compute levels for explosion starting from external demand roots
roots = sorted(external_demand["item"].unique())
levels = compute_levels(bom, roots)
items_ordered = sorted(items, key=lambda x: levels.get(x, 10**6))
cm = children_map(bom)

# Capacity tracking per plant-line per week (remaining hours)
capacity = {plant: {line: {w: plants[plant]["lines"][line] for w in weeks} for line in plants[plant]["lines"]} for plant in plants}


print(items_ordered)
print(roots)
print(levels)
print(cm)
print(capacity)



['FP', 'A', 'B', 'RawX', 'RawY']
['FP']
{'FP': 0, 'A': 1, 'B': 1, 'RawX': 2, 'RawY': 2}
defaultdict(<class 'list'>, {'FP': [('A', 2), ('B', 3)], 'A': [('RawX', 1)], 'B': [('RawY', 2)]})
{'Plant1': {'LineA1': {0: 40, 1: 40, 2: 40, 3: 40, 4: 40, 5: 40, 6: 40, 7: 40, 8: 40}, 'LineA2': {0: 30, 1: 30, 2: 30, 3: 30, 4: 30, 5: 30, 6: 30, 7: 30, 8: 30}}, 'Plant2': {'LineB1': {0: 50, 1: 50, 2: 50, 3: 50, 4: 50, 5: 50, 6: 50, 7: 50, 8: 50}}}


In [ ]:

# Function to allocate production to plant lines respecting capacity.
def allocate_production(plant, item, week, qty):
    """
    Attempt to allocate production of `qty` units of `item` at `plant` in `week`.
    Returns allocated_qty (may be less than requested) and unallocated_qty.
    Allocates across lines in arbitrary order, honoring available hours. 
    """
    if item not in production_sites or plant not in production_sites.get(item, []):
        return 0, qty  # can't produce here
    hours_per_unit = time_per_unit.get(item, 0)
    if hours_per_unit == 0:
        # Purchased item - no capacity required; treat as fully allocated
        return qty, 0
    remaining = qty
    allocated = 0
    
    for line, week_caps in capacity[plant].items():
        avail_hours = week_caps.get(week, 0)
        if avail_hours <= 0:
            continue
        max_units = int(avail_hours // hours_per_unit)
        if max_units <= 0:
            continue
        take = min(max_units, remaining)
        # consume hours
        used_hours = take * hours_per_unit
        capacity[plant][line][week] -= used_hours
        allocated += take
        remaining -= take
        if remaining <= 0:
            break
    return allocated, remaining

# Planning loop by levels, then by plant and time
for it in items_ordered:
    lt = int(lead_time.get(it, 0))
    # For each plant that could have demand/inventory (iterate plants)
    for plant in plants.keys():
        df = tp[plant][it]
        # iterate weeks
        for w in weeks:
            gross = df.loc[w, "Gross Req"]
            sched = df.loc[w, "Scheduled Rcpt"]
            onhand_prev = df.loc[w-1, "Proj On-Hand"] if w > 0 else df.loc[0, "Proj On-Hand"]
            available = onhand_prev + sched

            if available >= gross:
                net = 0
                planned_rcpt = 0
                proj_onhand = available - gross
            else:
                net = gross - available
                # Determine candidate planned receipt location(s)
                # For produced items: try to produce in eligible production sites (prefer same plant)
                planned_rcpt = 0
                unfilled = net

                # If item is producible in this plant, attempt allocation for receipt at w (means production finished in w)
                if it in production_sites and plant in production_sites[it]:
                    alloc, rem = allocate_production(plant, it, w, unfilled)
                    planned_rcpt += alloc
                    unfilled = rem

                # For purchased items or remaining unfilled -> treat as external purchase arriving (no capacity) at w
                if unfilled > 0 and time_per_unit.get(it, 0) == 0:
                    # purchased items can be received fully
                    planned_rcpt += unfilled
                    unfilled = 0
                # Any leftover unfilled becomes backorder (not planned)
                proj_onhand = 0 if planned_rcpt + available < gross else (available + planned_rcpt - gross)
                # Note planned_rcpt may be partial if capacity constrained
                # Update net accordingly (the unfilled remains as Net Req but not planned receipt)
                net = max(gross - (available + planned_rcpt), 0)

            df.loc[w, "Net Req"] = net
            df.loc[w, "Planned Rcpt"] = planned_rcpt
            df.loc[w, "Proj On-Hand"] = proj_onhand

            # Time-phase: releases are receipts shifted by lead time leftwards
            rel_week = w - lt
            if planned_rcpt > 0 and rel_week >= 0:
                df.loc[rel_week, "Planned Rel"] += planned_rcpt

        # After computing per-plant for this item, drive children's gross requirements
        if it in cm:
            for w in weeks:
                parent_rel = df.loc[w, "Planned Rel"]
                if parent_rel > 0:
                    for child, qty_per in cm[it]:
                        # Determine child demand distribution across plants:
                        # Simple rule: if child is producible, route to plants that produce it; 
                        # otherwise, assume procurement happens at same plant as parent
                        if child in production_sites:
                            # try to push child gross req to plants that produce it
                            sites = production_sites[child]
                            # distribute proportionally across sites (simple equal split)
                            qty_each = parent_rel * qty_per // len(sites)
                            remainder = parent_rel * qty_per - qty_each * len(sites)
                            for idx, psite in enumerate(sites):
                                add_qty = qty_each + (1 if idx < remainder else 0)
                                if add_qty > 0 and psite in tp:
                                    tp[psite][child].loc[w, "Gross Req"] += add_qty
                        else:
                            # purchased: route to same plant as parent
                            tp[plant][child].loc[w, "Gross Req"] += parent_rel * qty_per

## Prepare output

In [13]:
# ----------------------------
# Build outputs
# ----------------------------
# Flatten per-plant-item tables to combined dataframe for display
frames = []
for plant in tp:
    for it in items_ordered:
        tmp = tp[plant][it].copy()
        tmp = tmp.reset_index().rename(columns={"index": "Week"})
        tmp.insert(0, "Plant", plant)
        tmp.insert(1, "Item", it)
        frames.append(tmp)
combined = pd.concat(frames, ignore_index=True)


# Also show capacity utilization table per plant-line-week
cap_frames = []
for plant in capacity:
    for line in capacity[plant]:
        for w in weeks:
            cap_frames.append({"Plant": plant, "Line": line, "Week": w, "RemainingHours": capacity[plant][line][w]})
cap_df = pd.DataFrame(cap_frames)


# Save to Excel
excel_path = "2nd_mrp_multi_plant_capacity.xlsx"
with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    combined.to_excel(writer, sheet_name="MRP_Grid", index=False)
    cap_df.to_excel(writer, sheet_name="Capacity_Remaining", index=False)

excel_path

'2nd_mrp_multi_plant_capacity.xlsx'

# 3. (chưa chạm) With safety stock, lot-sizing policies, MOQ, rounding, backordering/rescheduling logic

adding:
- a
- a
- b
- c
- d


## config

In [ ]:
# Third attempt to run Single-plant MRP with Rolling 7-day aggregation and Weekly bucket options.
import math
import pandas as pd
from collections import defaultdict, deque


# ----------------------------
# CONFIG
# ----------------------------
horizon_days = 30  # planning horizon days 0..30 (inclusive)
days = list(range(horizon_days + 1))

plant = "Plant1"

external_demand = pd.DataFrame([
    {"item": "FP", "day": 10, "qty": 120},
    {"item": "FP", "day": 12, "qty": 60},
    {"item": "A",  "day": 8,  "qty": 40},
    {"item": "A",  "day": 15, "qty": 30},
])

bom = pd.DataFrame([
    {"parent": "FP", "component": "A",    "qty_per": 2},
    {"parent": "FP", "component": "B",    "qty_per": 3},
    {"parent": "A",  "component": "RawX", "qty_per": 1},
    {"parent": "B",  "component": "RawY", "qty_per": 2},
])

lead_time = {"FP": 7, "A": 5, "B": 10, "RawX": 14, "RawY": 7}
on_hand0 = {"FP": 0, "A": 20, "B": 10, "RawX": 50, "RawY": 0}
safety_stock = {"FP": 10, "A": 5, "B": 5, "RawX": 20, "RawY": 0}

lot_policy = {"FP": "ROLLING_7D", "A": "L4L", "B": "MULTIPLE", "RawX": "L4L", "RawY": "L4L"}
eoq_params = {"FP": {"setup_cost": 100.0, "holding_cost_day": 0.02}}
min_max_params = {"B": {"min": 100, "max": 500}}
min_multiples = {"FP": 50, "A": 10, "B": 10, "RawX": 100, "RawY": 50}
min_order_qty = {"FP": 50, "A": 10, "B": 50, "RawX": 100, "RawY": 50}

lines = {"Line1": 8*8, "Line2": 6*8}
time_per_unit = {"FP": 0.5, "A": 0.2, "B": 0.3, "RawX": 0.0, "RawY": 0.0}
production_items = {"FP", "A", "B"}

# ----------------------------
# HELPERS
# ----------------------------
def all_items(bom, external):
    items = set(external["item"].unique())
    items |= set(bom["parent"].unique())
    items |= set(bom["component"].unique())
    return sorted(items)

def children_map(bom):
    cm = defaultdict(list)
    for _, r in bom.iterrows():
        cm[r["parent"]].append((r["component"], r["qty_per"]))
    return cm

def week_of_day(d):
    return d // 7

def eoq_quantity(item, daily_demand_est):
    params = eoq_params.get(item)
    if not params or daily_demand_est <= 0:
        return None
    annual_demand = daily_demand_est * 365
    setup = params["setup_cost"]
    holding_daily = params["holding_cost_day"]
    annual_hold = holding_daily * 365
    q = int(max(1, round(((2 * setup * annual_demand) / annual_hold) ** 0.5)))
    return q

def round_up_to_multiple(q, multiple):
    if multiple <= 1:
        return q
    return int(((q + multiple - 1) // multiple) * multiple)

capacity = {line: {d: lines[line] for d in days} for line in lines}

def allocate_capacity(item, day, qty):
    if time_per_unit.get(item, 0) == 0 or item not in production_items:
        return qty, 0
    hours = time_per_unit[item]
    remaining = qty
    allocated = 0
    for line in capacity:
        avail_hours = capacity[line].get(day, 0)
        if avail_hours <= 0:
            continue
        max_units = int(avail_hours // hours)
        if max_units <= 0:
            continue
        take = min(max_units, remaining)
        used = take * hours
        capacity[line][day] -= used
        allocated += take
        remaining -= take
        if remaining <= 0:
            break
    return allocated, remaining

# ----------------------------
# INIT
# ----------------------------
items = all_items(bom, external_demand)
columns = ["Gross Req", "Scheduled Rcpt", "Proj On-Hand", "Net Req", "Planned Rcpt", "Planned Rel", "Unfilled"]
tp = {it: pd.DataFrame(0, index=days, columns=columns) for it in items}

for _, r in external_demand.iterrows():
    it = r["item"]
    d = int(r["day"])
    tp[it].loc[d, "Gross Req"] += int(r["qty"])

for it, qty in on_hand0.items():
    if it in tp:
        tp[it].loc[0, "Proj On-Hand"] = qty

cm = children_map(bom)

daily_est = {}
for it in items:
    total = int(tp[it]["Gross Req"].sum())
    daily_est[it] = max(1, int(total / max(1, horizon_days)))

covered = {it: set() for it in items}

# compute simple levels
levels = {it: 0 for it in items}
q = deque(sorted([r for r in external_demand["item"].unique()]))
while q:
    p = q.popleft()
    for c, _ in cm.get(p, []):
        if c not in levels or levels[c] < levels.get(p, 0) + 1:
            levels[c] = levels.get(p, 0) + 1
            q.append(c)
items_ordered = sorted(items, key=lambda x: levels.get(x, 0))




'output_python_example/mrp_single_plant_rolling_weekly_v3.xlsx'

## main loop

In [ ]:
# ----------------------------
# MAIN PLANNING LOOP (no rescheduling/backordering)
# ----------------------------
for it in items_ordered:
    lt = int(lead_time.get(it, 0))
    policy = lot_policy.get(it, "L4L")
    df = tp[it]
    for d in days:
        gross = df.loc[d, "Gross Req"]
        sched = df.loc[d, "Scheduled Rcpt"]
        onhand_prev = df.loc[d-1, "Proj On-Hand"] if d > 0 else df.loc[0, "Proj On-Hand"]
        available = onhand_prev + sched

        ss = safety_stock.get(it, 0)
        already = (d in covered[it])

        if available >= gross + ss:
            net = 0
            planned_rcpt = 0
            proj_onhand = available - gross
        else:
            net = max(gross + ss - available, 0)
            planned_rcpt = 0
            proj_onhand = max(0, available - ss)

            if net > 0 and not already:
                if policy == "L4L":
                    target = net
                elif policy == "EOQ":
                    q_eoq = eoq_quantity(it, daily_est.get(it, 1))
                    target = q_eoq if q_eoq and q_eoq >= net else net
                elif policy == "MINMAX":
                    params = min_max_params.get(it, {})
                    target = max(net, params.get("min", net))
                    if params.get("max"):
                        target = min(target, params.get("max"))
                elif policy == "MULTIPLE":
                    target = max(net, min_order_qty.get(it, 1))
                elif policy == "ROLLING_7D":
                    agg = 0
                    for dd in range(d, min(horizon_days+1, d+7)):
                        if dd not in covered[it]:
                            agg += tp[it].loc[dd, "Gross Req"]
                    target = max(agg - max(0, onhand_prev - ss), 0)
                elif policy == "WEEKLY_BUCKET":
                    wk = week_of_day(d)
                    agg = 0
                    for dd in range(wk*7, min(horizon_days+1, wk*7 + 7)):
                        if dd not in covered[it]:
                            agg += tp[it].loc[dd, "Gross Req"]
                    target = max(agg - max(0, onhand_prev - ss), 0)
                else:
                    target = net

                moq = min_order_qty.get(it, 1)
                mult = min_multiples.get(it, 1)
                target = max(target, moq)
                if mult > 1:
                    target = round_up_to_multiple(target, mult)

                allocated, unallocated = allocate_capacity(it, d, target)
                planned_rcpt = allocated
                if unallocated > 0:
                    df.loc[d, "Unfilled"] += unallocated

                # mark covered days for ROLLING_7D and WEEKLY_BUCKET
                if policy == "ROLLING_7D":
                    remaining_cover = target
                    for dd in range(d, min(horizon_days+1, d+7)):
                        req = tp[it].loc[dd, "Gross Req"]
                        if dd in covered[it]:
                            continue
                        if remaining_cover <= 0:
                            break
                        cover_qty = min(req, remaining_cover)
                        if cover_qty > 0:
                            covered[it].add(dd)
                            remaining_cover -= cover_qty
                if policy == "WEEKLY_BUCKET":
                    wk = week_of_day(d)
                    remaining_cover = target
                    for dd in range(wk*7, min(horizon_days+1, wk*7 + 7)):
                        req = tp[it].loc[dd, "Gross Req"]
                        if dd in covered[it]:
                            continue
                        if remaining_cover <= 0:
                            break
                        cover_qty = min(req, remaining_cover)
                        if cover_qty > 0:
                            covered[it].add(dd)
                            remaining_cover -= cover_qty

                df.loc[d, "Planned Rcpt"] += planned_rcpt
                rel_day = d - lt
                if rel_day >= 0:
                    df.loc[rel_day, "Planned Rel"] += planned_rcpt

                proj_onhand = max(available + planned_rcpt - gross, 0)

        df.loc[d, "Net Req"] = max(0, net)
        df.loc[d, "Proj On-Hand"] = proj_onhand

    # explode children based on Planned Rel
    for d in days:
        parent_rel = df.loc[d, "Planned Rel"]
        if parent_rel > 0 and it in cm:
            for child, qty_per in cm[it]:
                need_day = d
                if need_day <= horizon_days:
                    tp[child].loc[need_day, "Gross Req"] += int(parent_rel * qty_per)

# ----------------------------
# OUTPUT and save
# ----------------------------
frames = []
for it in items_ordered:
    tmp = tp[it].copy().reset_index().rename(columns={"index": "Day"})
    tmp.insert(0, "Plant", plant)
    tmp.insert(1, "Item", it)
    frames.append(tmp)
combined = pd.concat(frames, ignore_index=True)



# Save to Excel
excel_path = "output_python_example/mrp_single_plant_rolling_weekly_v3.xlsx"
with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    for it in items_ordered:
        df = combined[combined["Item"] == it].drop(columns=["Plant", "Item"]).set_index("Day")
        df.to_excel(writer, sheet_name=it)
    combined.to_excel(writer, sheet_name="All_Items", index=False)

excel_path

In [ ]:
def round_up_to_multiple(q, multiple):
    if multiple <= 1:
        return int(q)
    return int(((q + multiple - 1) // multiple) * multiple)



round_up_to_multiple(1000,50)

# Third - If split module

In [16]:
"""
Modular MRP functions: single-plant, daily buckets.
Functions:
 - calculate_net_requirement
 - calculate_lot_size
 - schedule_planned_receipt_and_release
 - explode_bom_children
 - driver: run_mrp_by_levels

Assumptions:
 - tp[item] is a pandas.DataFrame indexed by day with columns:
   ["Gross Req","Scheduled Rcpt","Proj On-Hand","Net Req","Planned Rcpt","Planned Rel","Unfilled"]
 - single-plant only
 - no rescheduling/backordering logic (unfilled is recorded)
"""

import math
import pandas as pd
from collections import defaultdict, deque

# -----------------------
# Helper / policy utils
# -----------------------
def round_up_to_multiple(q, multiple):
    if multiple <= 1:
        return int(q)
    return int(((q + multiple - 1) // multiple) * multiple)

def eoq_quantity(annual_demand, setup_cost, annual_holding_cost):
    if annual_demand <= 0 or annual_holding_cost <= 0:
        return None
    return int(max(1, round(((2 * setup_cost * annual_demand) / annual_holding_cost) ** 0.5)))


def date_range_index(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D")

## core function

In [ ]:
# -----------------------
# Core MRP functions
# -----------------------
def calculate_net_requirement(item, curr_date, tp_item_df, safety_stock=0):
    """
    Return (net_qty, available_before_receipts, onhand_before)
    - tp_item_df indexed by datetime.date or pd.Timestamp
    - safety_stock kept as a hard buffer (not consumed)
    """
    gross = int(tp_item_df.at[curr_date, "Gross Req"]) if curr_date in tp_item_df.index else 0
    sched = int(tp_item_df.at[curr_date, "Scheduled Rcpt"]) if curr_date in tp_item_df.index else 0
    # previous day on-hand
    idx = tp_item_df.index
    first_day = idx.min()
    if curr_date == first_day:
        onhand_prev = int(tp_item_df.at[first_day, "POH 1 (before LS)"])
    else:
        prev = curr_date - pd.Timedelta(days=1)
        # If prev not in index (shouldn't happen), fallback to first day
        if prev not in idx:
            prev = first_day
        onhand_prev = int(tp_item_df.at[prev, "POH 1 (before LS)"])
    available = onhand_prev + sched
    usable = max(0, available - safety_stock)  # cannot draw below safety stock
    net = 0 if usable >= gross else gross - usable
    return int(net), int(available), int(onhand_prev)

def calculate_lot_size(item, net_qty, policy, policy_params, daily_gross_series=None, current_date=None, horizon_end=None):
    """
    Return an order quantity according to policy.
    policy_params: dict for customization per item.
    - For ROLLING_7D: policy_params[item]['window_days'] default 7
    - For WEEKLY_BUCKET: aggregates calendar week (Mon-Sun) or sunday-start based on implementation (we use day //7 from start)
    """
    if net_qty <= 0:
        return 0

    params = policy_params.get(item, {})
    if policy == "L4L":
        target = net_qty

    elif policy == "MULTIPLE":
        mult = params.get("multiple", 1)
        moq = params.get("min_order_qty", 1)
        t = max(net_qty, moq)
        target = round_up_to_multiple(t, mult)

    elif policy == "MINMAX":
        mn = params.get("min_qty", 0)
        mx = params.get("max_qty", 10**12)
        target = max(net_qty, mn)
        target = min(target, mx)
        # apply multiple if present
        mult = params.get("multiple", 1)
        if mult > 1:
            target = round_up_to_multiple(target, mult)

    elif policy == "EOQ":
        # simple EOQ using provided annual demand or proxy by daily_gross_series scaled
        annual = params.get("annual_demand")
        setup = params.get("setup_cost", params.get("order_cost", None))
        annual_hold = params.get("annual_holding_cost", params.get("holding_cost", None))
        if annual is None and daily_gross_series is not None and horizon_end is not None and current_date is not None:
            # use sum over remaining horizon scaled to annual
            rem_series = daily_gross_series.loc[current_date:horizon_end]
            annual = int(rem_series.sum() * (365.0 / max(1, (horizon_end - current_date).days + 1)))
        if annual and setup and annual_hold:
            q = eoq_quantity(annual, setup, annual_hold)
            target = q if q >= net_qty else net_qty
        else:
            target = net_qty

        mult = params.get("multiple", 1)
        if mult > 1:
            target = round_up_to_multiple(target, mult)

    elif policy == "ROLLING_7D":
        # aggregate gross requirements in [current_date .. current_date + window -1]
        if daily_gross_series is None or current_date is None or horizon_end is None:
            target = net_qty
        else:
            window = params.get("window_days", 7)
            end = min(horizon_end, current_date + pd.Timedelta(days=window-1))
            agg = int(daily_gross_series.loc[current_date:end].sum())
            target = max(net_qty, agg)

            # apply multiples/moq
            mult = params.get("multiple", 1)
            moq = params.get("min_order_qty", 1)
            target = max(target, moq)
            if mult > 1:
                target = round_up_to_multiple(target, mult)

    elif policy == "WEEKLY_BUCKET":
        # compute week start based on first index of series (calendar week alignment optional)
        if daily_gross_series is None or current_date is None or horizon_end is None:
            target = net_qty
        else:
            # simple week as blocks of 7 days from start of index
            start = daily_gross_series.index.min()
            week_num = ((current_date - start).days) // 7
            week_start = start + pd.Timedelta(days=week_num*7)
            week_end = min(horizon_end, week_start + pd.Timedelta(days=6))
            agg = int(daily_gross_series.loc[week_start:week_end].sum())
            target = max(net_qty, agg)
            mult = params.get("multiple", 1)
            moq = params.get("min_order_qty", 1)
            target = max(target, moq)
            if mult > 1:
                target = round_up_to_multiple(target, mult)

    else:
        target = net_qty

    return int(max(0, target))

def schedule_planned_receipt_and_release(item, arrival_date, qty, tp_item_df, lead_time_map):
    """
    Write Planned Rcpt at arrival_date and Planned Rel at arrival_date - lead_time.
    Returns (release_date, arrival_date).
    """
    if qty <= 0:
        return None, None
    lt = int(lead_time_map.get(item, 0))
    release_date = arrival_date - pd.Timedelta(days=lt)
    # increment, safe for index bounds
    if arrival_date in tp_item_df.index:
        tp_item_df.at[arrival_date, "Planned Rcpt"] += int(qty)
    # planned release at release_date only if within horizon
    if release_date in tp_item_df.index:
        tp_item_df.at[release_date, "Planned Rel"] += int(qty)
    return release_date, arrival_date

def explode_bom_children(parent_item, rel_date, qty_released, bom_df, tp_map):
    """
    Single-level BOM explosion: for each child, add child's Gross Req on rel_date by qty_released * qty_per.
    """
    children = bom_df[bom_df["parent"] == parent_item]
    for _, r in children.iterrows():
        child = r["component"]
        qty_per = float(r["qty_per"])
        if rel_date in tp_map[child].index:
            tp_map[child].at[rel_date, "Gross Req"] += int(qty_released * qty_per)
            




## core mrp logic

In [18]:
def run_mrp_by_levels(bom_df, items_ordered, tp_map, lead_time_map,
                      lot_policy_map, lot_policy_params, safety_stock_map):
    """
    items_ordered: list top-down (parents before children)
    tp_map: dict item -> DataFrame with index = pd.DatetimeIndex daily and columns:
        ["Gross Req","Scheduled Rcpt","Proj On-Hand","Net Req","Planned Rcpt","Planned Rel","Unfilled"]
    lot_policy_map: dict item -> policy string
    lot_policy_params: dict item -> dict of policy parameters
    safety_stock_map: dict item -> int
    Returns: tp_map updated in-place
    """
    # Precompute daily series and horizon boundaries for each item
    daily_series = {it: tp_map[it]["Gross Req"] for it in tp_map}
    horizon_start = min(df.index.min() for df in tp_map.values())
    horizon_end = max(df.index.max() for df in tp_map.values())

    # Tracking covered days for rolling aggregation to avoid double coverage (per item)
    covered_days = {it: set() for it in tp_map}

    for item in items_ordered:
        df = tp_map[item]
        policy = lot_policy_map.get(item, "L4L")
        params_for_item = lot_policy_params.get(item, {})
        for curr_date in df.index:
            # 1) Net requirement
            net, available, onhand_prev = calculate_net_requirement(item, curr_date, df, safety_stock=safety_stock_map.get(item, 0))
            df.at[curr_date, "Net Req"] = int(net)
            if net <= 0:
                # update projected on-hand: available - gross (but keep >=0)
                df.at[curr_date, "Proj On-Hand"] = int(max(0, available - df.at[curr_date, "Gross Req"]))
                continue

            # If date already covered by earlier aggregation (used by rolling/week policies), skip creating another order
            if curr_date in covered_days[item]:
                # No action (net remains but expected to be covered by earlier planned receipt)
                df.at[curr_date, "Proj On-Hand"] = int(max(0, available - df.at[curr_date, "Gross Req"]))
                continue

            # 2) Lot size calculation
            # Inject current_date & horizon_end into params for policies that depend on them
            pparams = dict(params_for_item)  # shallow copy to avoid mutation
            pparams["_current_date"] = curr_date
            pparams["_horizon_end"] = horizon_end
            order_qty = calculate_lot_size(item=item, net_qty=net, policy=policy,
                                          policy_params=lot_policy_params,  # pass all params (function reads item key)
                                          daily_gross_series=daily_series.get(item),
                                          current_date=curr_date,
                                          horizon_end=horizon_end)
            if order_qty <= 0:
                df.at[curr_date, "Proj On-Hand"] = int(max(0, available - df.at[curr_date, "Gross Req"]))
                continue

            # 3) Scheduling (receipt at curr_date, release = curr_date - lead_time)
            rel_date, arr_date = schedule_planned_receipt_and_release(item, curr_date, order_qty, df, lead_time_map)

            # If order scheduled, update projected on-hand for arrival day after consumption
            planned_rcpt_at_day = int(df.at[curr_date, "Planned Rcpt"]) if curr_date in df.index else 0
            df.at[curr_date, "Proj On-Hand"] = int(max(0, available + planned_rcpt_at_day - df.at[curr_date, "Gross Req"]))

            # 4) Explosion: use planned release (rel_date) to create child gross requirements
            # Mark covered days for ROLLING_7D/WEEKLY_BUCKET to prevent duplicate aggregation
            if policy == "ROLLING_7D":
                window = lot_policy_params.get(item, {}).get("window_days", 7)
                # cover days curr_date .. curr_date+window-1 (bounded by horizon_end)
                end_cover = min(horizon_end, curr_date + pd.Timedelta(days=window-1))
                remaining = order_qty
                # We mark days as covered only to the extent they are consumed; simple approach: mark whole days in window.
                for day in pd.date_range(curr_date, end_cover, freq="D"):
                    covered_days[item].add(day)
            elif policy == "WEEKLY_BUCKET":
                # compute week block from horizon_start
                week_idx = ((curr_date - horizon_start).days) // 7
                week_start = horizon_start + pd.Timedelta(days=week_idx*7)
                week_end = min(horizon_end, week_start + pd.Timedelta(days=6))
                for day in pd.date_range(week_start, week_end, freq="D"):
                    covered_days[item].add(day)

            if rel_date is not None:
                explode_bom_children(parent_item=item, rel_date=rel_date, qty_released=order_qty, bom_df=bom_df, tp_map=tp_map)

    return tp_map

## implementation the code

In [19]:
# -----------------------
# Example: build inputs (toy dataset); do NOT run here if you only want code.
# -----------------------
if __name__ == "__main__":
    # Example config (dates as day-level)
    start = pd.Timestamp("2025-10-01")
    end = pd.Timestamp("2025-10-20")
    idx = date_range_index(start, end)

    # BOM example (multi-level)
    bom_df = pd.DataFrame([
        {"parent": "A", "component": "B", "qty_per": 2},
        {"parent": "A", "component": "C", "qty_per": 1},
        {"parent": "B", "component": "D", "qty_per": 3},
    ])

    # Policies per item
    lot_policy_map = {
        "A": "ROLLING_7D",
        "B": "WEEKLY_BUCKET",
        "C": "L4L",
        "D": "L4L",
    }
    lot_policy_params = {
        "A": {"window_days": 7, "multiple": 200, "min_order_qty": 200},  # ROLLING_7D -> 7d, MOQ/multiple
        "B": {"multiple": 100, "min_order_qty": 100},
        "C": {},
        "D": {},
    }

    # Lead times (days)
    lead_time_map = {"A": 2, "B": 1, "C": 1, "D": 1}

    # Starting on-hand and safety stock
    on_hand0 = {"A": 100, "B": 0, "C": 0, "D": 0}
    safety_stock = {"A": 50, "B": 0, "C": 0, "D": 0}

    # Demand (Gross requirements) for finished product A on specific dates
    demand_records = [
        {"item": "A", "date": pd.Timestamp("2025-10-01"), "qty": 50},
        {"item": "A", "date": pd.Timestamp("2025-10-04"), "qty": 20},
        {"item": "A", "date": pd.Timestamp("2025-10-09"), "qty": 30},
    ]

    # Build tp_map: empty time-phased DataFrame per item
    items = sorted(set(bom_df["parent"]).union(set(bom_df["component"])).union({r["item"] for r in demand_records}))
    tp_map = {}
    cols = ["Gross Req", "Scheduled Rcpt", "Proj On-Hand", "Net Req", "Planned Rcpt", "Planned Rel", "Unfilled"]
    for it in items:
        df = pd.DataFrame(0, index=idx, columns=cols)
        # seed on-hand at first day
        df.at[idx[0], "Proj On-Hand"] = int(on_hand0.get(it, 0))
        tp_map[it] = df

    # Seed gross requirements from demand_records (MPS)
    for rec in demand_records:
        it = rec["item"]
        d = rec["date"]
        if d in tp_map[it].index:
            tp_map[it].at[d, "Gross Req"] += int(rec["qty"])

    # Determine order of items by BOM level (parents first)
    def compute_levels(bom):
        children = defaultdict(list)
        parents = set()
        for _, r in bom.iterrows():
            parents.add(r["parent"])
            children[r["parent"]].append(r["component"])
        level = {}
        q = deque(sorted([r["item"] for r in demand_records]))
        for r in q:
            level[r] = 0
        while q:
            p = q.popleft()
            for c in children.get(p, []):
                if c not in level or level[c] < level[p] + 1:
                    level[c] = level.get(p, 0) + 1
                    q.append(c)
        # produce parents first (lower level number first)
        ordered = sorted(list(set(items)), key=lambda x: level.get(x, 0))
        return ordered

    items_ordered = compute_levels(bom_df)

    # Run MRP
    run_mrp_by_levels(bom_df=bom_df, items_ordered=items_ordered, tp_map=tp_map,
                      lead_time_map=lead_time_map, lot_policy_map=lot_policy_map,
                      lot_policy_params=lot_policy_params, safety_stock_map=safety_stock)

    # Produce combined view for review
    frames = []
    for it in items_ordered:
        t = tp_map[it].copy().reset_index().rename(columns={"index": "Date"})
        t.insert(0, "Item", it)
        frames.append(t)
    combined = pd.concat(frames, ignore_index=True)

    # Print short sample to console
    print(combined[combined["Item"] == "A"].head(20))

    # To export to Excel:
    # combined.to_excel("mrp_output.xlsx", index=False)

   Item       Date  Gross Req  Scheduled Rcpt  Proj On-Hand  Net Req  \
0     A 2025-10-01         50               0            50        0   
1     A 2025-10-02          0               0            50        0   
2     A 2025-10-03          0               0            50        0   
3     A 2025-10-04         20               0           230       20   
4     A 2025-10-05          0               0           230        0   
5     A 2025-10-06          0               0           230        0   
6     A 2025-10-07          0               0           230        0   
7     A 2025-10-08          0               0           230        0   
8     A 2025-10-09         30               0           200        0   
9     A 2025-10-10          0               0           200        0   
10    A 2025-10-11          0               0           200        0   
11    A 2025-10-12          0               0           200        0   
12    A 2025-10-13          0               0           200     